# FreshNet GPU training (Colab)

Use this when local hardware is CPU-only or too slow. Upload/mount the repo (or at least `ml/`, `configs/train.yaml`, and `data/processed/manifests/`).

**Do not invent metrics.** Keep the frozen split fingerprint:
`005d750025b30ce1074333714f7e2ee7b89cc0ec9acd4ba56f62010edd5850f9`

Runtime → GPU. Train context → detail → fused. Evaluate on validation first; test only after selection is frozen.

In [ ]:
import torch, torchvision, sys
print(sys.version)
print('torch', torch.__version__, 'torchvision', torchvision.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

In [ ]:
# If the repo is not already present, upload a zip or clone your private copy, then:
# %cd /content/FreshNet
import json
from pathlib import Path
fp=json.loads(Path('data/processed/manifests/split_fingerprint.json').read_text())['split_fingerprint']
assert fp=='005d750025b30ce1074333714f7e2ee7b89cc0ec9acd4ba56f62010edd5850f9'
print('fingerprint_ok', fp)
%pip install -q matplotlib pyyaml pillow

In [ ]:
import os
os.environ['PYTHONPATH']='ml'
!python ml/train_model.py --architecture context --config configs/train.yaml --device cuda
!python ml/train_model.py --architecture detail --config configs/train.yaml --device cuda
!python ml/train_model.py --architecture fused --config configs/train.yaml --device cuda

In [ ]:
!python ml/evaluate_model.py --checkpoint artifacts/checkpoints/context/best.pt --split val
!python ml/evaluate_model.py --checkpoint artifacts/checkpoints/detail/best.pt --split val
!python ml/evaluate_model.py --checkpoint artifacts/checkpoints/fused/best.pt --split val
print('Select deployment model from validation metrics, then evaluate test once.')